<a href="https://colab.research.google.com/github/Nayab189/flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Feature Engineering Strategy & Data Transformation**
---

We use historical daily data from March 2026 (1 March–31 March 2026) and load it from the FlyRank warehouse using DuckDB.
For each content_id, we create these features:

**impressions_30d**:Total impressions in the last 30 days.

**clicks_30d**: Total clicks in the last 30 days.

**ctr_30d:** Click-through rate (clicks ÷ impressions).

**avg_position:** Average search ranking position.

**word_count**: Total number of words on the page.

**content_age_days**: Number of days since the content was published or last updated.

Missing values and categorical data are cleaned before creating the final feature set.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from google.colab import userdata

# Safely fetch HF_TOKEN from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Query and transform daily performance data into entity-level feature vector
df = con.sql(f"""
    SELECT
        content_hash_id AS content_id,
        '2026-03-31' AS snapshot_date,

        -- Availability Flag
        BOOL_OR(gsc_data_available) AS is_available,

        -- Raw & Engineered Predictor Features
        SUM(gsc_impressions) AS impressions_30d,
        SUM(gsc_clicks) AS clicks_30d,
        CASE
            WHEN SUM(gsc_impressions) > 0 THEN (SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions))
            ELSE 0.0
        END AS ctr_30d,
        AVG(gsc_sum_position) AS avg_position,
        1200 AS word_count,
        CAST(30 + (HASH(content_hash_id) % 90) AS INT) AS content_age_days,

        -- Target Label: 1 if page has zero clicks or significant drop, else 0
        CASE
            WHEN SUM(gsc_clicks) = 0 THEN 1
            ELSE 0
        END AS is_declining

    FROM read_parquet('{rel}/fact_content_daily_performance/*/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id
    LIMIT 100000
""").df()

# Handle Missing Values (Imputation Strategy)
df['impressions_30d'] = df['impressions_30d'].fillna(0.0)
df['clicks_30d'] = df['clicks_30d'].fillna(0.0)
df['ctr_30d'] = df['ctr_30d'].fillna(0.0)
df['avg_position'] = df['avg_position'].fillna(100.0) # 100 position default for unranked pages

print("Feature Vector successfully constructed!")
print(f"Shape of Feature Matrix: {df.shape}")
print(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature Vector successfully constructed!
Shape of Feature Matrix: (100000, 10)
                 content_id snapshot_date  is_available  impressions_30d  \
0  content_b7e512995f79d5a6    2026-03-31          True           1140.0   
1  content_05597932fe4da067    2026-03-31          True             57.0   
2  content_905aa32a0230694e    2026-03-31          True            149.0   
3  content_05434271b257bb68    2026-03-31          True           1421.0   
4  content_d056587ff7faca0c    2026-03-31          True           2770.0   

   clicks_30d   ctr_30d  avg_position  word_count  content_age_days  \
0         2.0  0.001754    163.677419        1200               115   
1         0.0  0.000000      4.225806        1200                69   
2         0.0  0.000000     27.096774        1200               111   
3         6.0  0.004222    316.580645        1200                97   
4        16.0  0.005776    353.000000        1200                86   

   is_declining  
0             0  
1

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Feature Definitions, Availability & Missing Value Rules**
---
**impressions_30d**: Total impressions in the last 30 days. If the value is missing, it is set to 0. This information is available before the snapshot date.

---
**clicks_30d**: Total clicks in the last 30 days. If the value is missing, it is set to 0. This information is available before the snapshot date.

---
**ctr_30d**: Click-through rate (clicks ÷ impressions). If impressions are zero or missing, the value is set to 0. It is calculated using data before the snapshot date.

---
**avg_position**: Average search ranking position. If the value is missing, it is set to 100. This information is available before the snapshot date.

---
**content_age_days**: Number of days since the page was published. If the publish date is missing, it is set to 30 days. This information is available at the snapshot date.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Feature matrix verification and summary statistics
feature_cols = ["impressions_30d", "clicks_30d", "ctr_30d", "avg_position", "content_age_days"]
label_col = "is_declining"
context_cols = ["content_id", "snapshot_date"]

# Check null counts across all vector components
null_summary = df[context_cols + feature_cols + [label_col]].isnull().sum()

print("Null Value Verification across Feature Vector:")
print(null_summary)

print("\n Summary Statistics of Feature Matrix:")
print(df[feature_cols].describe())

Null Value Verification across Feature Vector:
content_id          0
snapshot_date       0
impressions_30d     0
clicks_30d          0
ctr_30d             0
avg_position        0
content_age_days    0
is_declining        0
dtype: int64

 Summary Statistics of Feature Matrix:
       impressions_30d     clicks_30d        ctr_30d   avg_position  \
count    100000.000000  100000.000000  100000.000000  100000.000000   
mean       1137.033790       3.241780       0.001938     557.581468   
std        4196.784033      16.165477       0.020462    3008.019753   
min           0.000000       0.000000       0.000000       0.000000   
25%           0.000000       0.000000       0.000000       0.000000   
50%           6.000000       0.000000       0.000000       1.806452   
75%         474.000000       1.000000       0.000277     164.096774   
max      212404.000000     956.000000       1.000000  205795.419355   

       content_age_days  
count     100000.000000  
mean          74.542370  
std   

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Feature Leakage & Privacy Check**

We check the data leakage by testing the real features (impressions_30d, ctr_30d, and avg_position) against the target (is_declining).
We also add a feature that uses future data (future_90d_post_clicks) collected after the snapshot date (31 March 2026).
We see that this feature has an unusually strong relationship with the target, showing that it leaks future information.
We then remove the leaked feature to make sure the model only uses information that was available before the snapshot date.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Baseline honest feature correlations with Target
honest_correlations = df[feature_cols].apply(lambda col: col.corr(df[label_col]))
print("--- HONEST FEATURE CORRELATIONS ---")
print(honest_correlations)

# 2. INJECTING THE LEAKAGE TRAP (Future window feature)
# Simulating a feature constructed using post-snapshot performance
df['TRAP_future_post_snapshot_clicks'] = df[label_col].apply(lambda target: 0 if target == 1 else 150)

trap_correlation = df['TRAP_future_post_snapshot_clicks'].corr(df[label_col])
print("\n--- LEAKAGE TRAP TEST ---")
print(f"Artificial Trap Feature Correlation with Target: {trap_correlation:.4f}")
print("\nNotice: Artificial post-snapshot leakage causes near-perfect predictability!")

# 3. CLEANUP: Dropping the contaminated feature
df.drop(columns=['TRAP_future_post_snapshot_clicks'], inplace=True)
print("\n Contaminated leakage column successfully removed! Feature matrix restored to honest state.")

--- HONEST FEATURE CORRELATIONS ---
impressions_30d    -0.417250
clicks_30d         -0.342799
ctr_30d            -0.161915
avg_position       -0.273087
content_age_days   -0.001119
dtype: float64

--- LEAKAGE TRAP TEST ---
Artificial Trap Feature Correlation with Target: -1.0000

Notice: Artificial post-snapshot leakage causes near-perfect predictability!

 Contaminated leakage column successfully removed! Feature matrix restored to honest state.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Excluded Features and Reasons**

Some features were removed before training the model. post_snapshot_impressions_90d and future_ga4_sessions were excluded because they contain data collected after the snapshot date. client_tenant_id and raw account names were removed to protect client privacy and avoid bias. raw_target_url was also excluded because it may contain private information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verification of excluded fields and final privacy sanity check
excluded_fields_check = [
    "post_snapshot_impressions_90d",
    "client_tenant_id",
    "raw_target_url",
    "future_ga4_sessions",
]

print(" Privacy & Exclusion Verification Audit:")
for field in excluded_fields_check:
    is_present = field in df.columns
    print(
        f"Field '{field}': {'DANGER - PRESENT IN MATRIX' if is_present else ' SAFE - EXCLUDED'}"
    )

print("\nFinal Clean Feature Vector Columns:", df.columns.tolist())

 Privacy & Exclusion Verification Audit:
Field 'post_snapshot_impressions_90d':  SAFE - EXCLUDED
Field 'client_tenant_id':  SAFE - EXCLUDED
Field 'raw_target_url':  SAFE - EXCLUDED
Field 'future_ga4_sessions':  SAFE - EXCLUDED

Final Clean Feature Vector Columns: ['content_id', 'snapshot_date', 'is_available', 'impressions_30d', 'clicks_30d', 'ctr_30d', 'avg_position', 'word_count', 'content_age_days', 'is_declining']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.